<a href="https://colab.research.google.com/github/Arooba93/MyProjects/blob/main/Hugging_face_mamba_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Necessary Packages
First, we install `transformers`, `datasets`, `evaluate`, `accelerate`, and `timm` (often required by vision models).

In [ ]:
!pip install -q ninja packaging wheel
!pip install -q transformers datasets evaluate accelerate timm scikit-learn
!pip install -q causal-conv1d mamba-ssm --no-build-isolation

  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 42.0 MB/s eta 0:00:00


### 2. Load the Dataset
We load the `Chanura04/CT-Scan-Brain-Tumor` dataset from Hugging Face.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Chanura04/CT-Scan-Brain-Tumor")

# Shuffle and select 1200 random samples from the train split
if 'train' in dataset:
    dataset['train'] = dataset['train'].shuffle(seed=42).select(range(min(1200, len(dataset['train']))))

display(dataset)

# Inspect the features to get the label names
if 'train' in dataset:
    labels = dataset['train'].features['label'].names
    print("Classes:", labels)
else:
    print("Please verify dataset splits and structure.")

Resolving data files:   0%|          | 0/4618 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 1200
    })
})

Classes: ['NORMAL', 'Tumor']


### 3. Preprocess the Data
We need to apply standard image transformations (resizing, normalizing) as expected by MambaVision.

In [ ]:
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor, Resize, CenterCrop

model_id = "nvidia/MambaVision-T-1K"

# The model repo lacks a preprocessor_config.json, so we manually define standard ImageNet stats
image_mean = [0.485, 0.456, 0.406]
image_std = [0.229, 0.224, 0.225]
size = (224, 224)

# Define torchvision transforms
normalize = Normalize(mean=image_mean, std=image_std)

_transforms = Compose([
    Resize(size),
    CenterCrop(size),
    ToTensor(),
    normalize
])

def transforms(examples):
    # Ensure the images are in RGB format
    examples["pixel_values"] = [_transforms(img.convert("RGB")) for img in examples["image"]]
    return examples

# Apply transformations on the fly
dataset = dataset.with_transform(transforms)

### 4. Load the Mamba Model
We initialize `AutoModelForImageClassification` using the Nvidia MambaVision model.

In [ ]:
from transformers import AutoModelForImageClassification, AutoConfig
import transformers
import importlib
import transformers.modeling_utils

# Reload the module to clear any stacked recursive patches from the kernel state
importlib.reload(transformers.modeling_utils)

# Now apply the safe patch exactly once
original_move = transformers.modeling_utils.PreTrainedModel._move_missing_keys_from_meta_to_device

def _patched_move(self, *args, **kwargs):
    # Override the read-only property on the class level to return a dict
    cls = type(self)
    if hasattr(cls, 'all_tied_weights_keys'):
        cls.all_tied_weights_keys = property(lambda self: {})
    return original_move(self, *args, **kwargs)

transformers.modeling_utils.PreTrainedModel._move_missing_keys_from_meta_to_device = _patched_move

# Create mappings for the labels
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

# Load config and override the label mappings to avoid the mismatch
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
config.id2label = id2label
config.label2id = label2id
config.num_labels = len(labels)

# Load the model with the updated config
model = AutoModelForImageClassification.from_pretrained(
    model_id,
    config=config,
    ignore_mismatched_sizes=True,
    trust_remote_code=True
)

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

### 5. Fine-tune the Model
Set up the `Trainer` to train and evaluate the model.

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers.modeling_outputs import ImageClassifierOutput
import torch.nn as nn
import numpy as np
import evaluate
import torch

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

# If dataset doesn't have a specific test/validation split, we split it here
if 'test' not in dataset and 'validation' not in dataset:
    dataset = dataset['train'].train_test_split(test_size=0.15)
    train_dataset = dataset['train']
    eval_dataset = dataset['test']
else:
    train_dataset = dataset['train']
    eval_dataset = dataset.get('validation', dataset.get('test'))

# Custom data collator to extract only the tensors we need
def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

# Patch the model to be fully compatible with the Hugging Face Trainer
# Retrieve the original method directly from the class to avoid kernel state recursion
_original_forward = type(model).forward.__get__(model)

def _hf_compatible_forward(pixel_values=None, labels=None, **kwargs):
    # Pass the image tensor positionally to the original model's forward pass
    outputs = _original_forward(pixel_values)

    # Extract logits handling both dict and object returns
    if isinstance(outputs, dict) and "logits" in outputs:
        logits = outputs["logits"]
    elif hasattr(outputs, "logits"):
        logits = outputs.logits
    else:
        logits = outputs

    loss = None
    if labels is not None:
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)

    return ImageClassifierOutput(loss=loss, logits=logits)

model.forward = _hf_compatible_forward

training_args = TrainingArguments(
    output_dir="./mambavision-tumor-classification",
    remove_unused_columns=False,  # Keep columns so transform can access 'image'
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

# Start fine-tuning
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,6.070889,1.210720,0.933333
2,2.383820,0.957674,0.955556
3,1.734737,0.750107,0.950000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=192, training_loss=4.64732255670242, metrics={'train_runtime': 73.5447, 'train_samples_per_second': 41.607, 'train_steps_per_second': 2.611, 'total_flos': 0.0, 'train_loss': 4.64732255670242, 'epoch': 3.0})

In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define source and destination paths
source_dir = "./mambavision-tumor-classification"
dest_dir = "/content/drive/MyDrive/mambavision-tumor-classification"

# Copy the directory to Google Drive
shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True)
print(f"Model successfully saved to {dest_dir}")

Mounted at /content/drive
Model successfully saved to /content/drive/MyDrive/mambavision-tumor-classification
